# 🔔 AIOps — Session 8
# Noise Reduction & Alert Correlation

---

| | |
|---|---|
| **Course** | AIOps Engineering |
| **Lab** | Lab 8 |
| **Topics** | Alert Storms & Fatigue · Event Correlation · Dependency Mapping |
| **Duration** | ~90 minutes |
| **Platform** | Kubernetes (kubectl must be available in this JupyterLab environment) |

---

## 🎯 Learning Objectives

By completing this lab you will be able to:

1. **Explain** how a single infrastructure failure causes an *alert storm* across microservices
2. **Apply** fingerprinting and time-window deduplication to suppress duplicate alerts
3. **Implement** rule-based and ML-based event correlation to link related alerts
4. **Build** a service dependency map and use it for automated root cause analysis
5. **Compress** hundreds of alerts into a small set of actionable incidents

---

## 🗺️ Lab Architecture

We simulate a realistic **e-commerce platform** running on Kubernetes:

```
  [frontend] ──▶ [api-gateway] ──▶ [order-service]   ──▶ [inventory-service]  ← ROOT CAUSE
                              ├──▶ [payment-service]  ──▶ [fraud-service]
                              └──▶ [notification-service]
```

The lab simulates a **database failure** inside `inventory-service` that cascades upward, generating ~500 alerts in 5 minutes — a classic *alert storm*.

---

## 📋 Lab Phases

| Phase | Title | Key Concept |
|-------|-------|-------------|
| **0** | Environment Setup | Install libraries |
| **1** | Deploy Microservices on K8s | `kubectl apply` |
| **2** | Simulate an Alert Storm | Alert Fatigue |
| **3** | Deduplicate & Reduce Noise | Fingerprinting · Flap Detection |
| **4** | Correlate Events | Rule-Based · DBSCAN · Topology |
| **5** | Build Dependency Map + RCA | Graph Traversal |
| **6** | Create Incident Register | AIOps Compression |
| **7** | Cleanup | `kubectl delete` |

> ▶️ **Run each cell in order using `Shift + Enter`**

---
## Phase 0 — Environment Setup

Install Python libraries and verify `kubectl` connectivity.

> ⏱️ Takes ~60 seconds on first run.

In [ ]:
# ── Install required libraries ─────────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q \
    pandas numpy matplotlib seaborn networkx \
    scikit-learn plotly ipywidgets tabulate
print("✅ Libraries installed")

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import subprocess, json, time, hashlib, random, textwrap
from datetime import datetime, timedelta
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor':   '#ffffff',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        10,
})

print("✅ Imports ready")

In [ ]:
# ── Verify kubectl access ──────────────────────────────────────────────────────
# This confirms your JupyterLab environment can talk to the Kubernetes cluster.

def run(cmd, check=False):
    """Run a shell command, print output, return (stdout, returncode)."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0 and result.stderr.strip():
        print("STDERR:", result.stderr.strip())
    return result.stdout.strip(), result.returncode

print("=" * 55)
print("kubectl version (client):")
run("kubectl version --client --short 2>/dev/null || kubectl version --client")

print("\nCluster nodes:")
out, rc = run("kubectl get nodes")

if rc == 0:
    print("\n✅ kubectl is connected to the cluster — ready to proceed!")
else:
    print("\n❌ kubectl cannot reach the cluster.")
    print("   Check your KUBECONFIG or cluster credentials before continuing.")

---
## Phase 1 — Deploy the Microservices Stack on Kubernetes

We create a **dedicated namespace** and deploy 7 services that represent a real e-commerce architecture.

Each pod has:
- Prometheus scrape annotations (for alert simulation)
- Liveness and readiness probes (so K8s can detect failures)
- Resource limits (realistic constraints)

> ⏱️ Allow 30–60 seconds for pods to reach `Running` state.

In [ ]:
# ── Create the lab namespace ───────────────────────────────────────────────────
# All resources go into 'aiops-lab8' — easy to clean up at the end.

NAMESPACE = "aiops-lab8"

ns_yaml = f"""\
apiVersion: v1
kind: Namespace
metadata:
  name: {NAMESPACE}
  labels:
    lab: session8
    managed-by: aiops-lab
"""

with open("/tmp/lab8-ns.yaml", "w") as f:
    f.write(ns_yaml)

run(f"kubectl apply -f /tmp/lab8-ns.yaml")
print(f"\n✅ Namespace '{NAMESPACE}' ready")

In [ ]:
# ── Service definitions ────────────────────────────────────────────────────────
# Each entry: (name, port, replicas, tier)

SERVICES = [
    ("frontend",             3000, 2, "frontend"),
    ("api-gateway",          8080, 2, "gateway"),
    ("order-service",        8081, 2, "backend"),
    ("payment-service",      8082, 2, "backend"),
    ("inventory-service",    8083, 1, "backend"),
    ("fraud-service",        8084, 1, "backend"),
    ("notification-service", 8085, 1, "backend"),
]

# ── Service dependency graph (caller → callee) ─────────────────────────────────
DEPENDENCY_GRAPH = {
    "frontend":             ["api-gateway"],
    "api-gateway":          ["order-service", "payment-service", "notification-service"],
    "order-service":        ["inventory-service"],
    "payment-service":      ["fraud-service"],
    "inventory-service":    [],
    "fraud-service":        [],
    "notification-service": [],
}

def build_manifest(name, port, replicas, tier):
    return f"""
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {name}
  namespace: {NAMESPACE}
  labels:
    app: {name}
    tier: {tier}
spec:
  replicas: {replicas}
  selector:
    matchLabels:
      app: {name}
  template:
    metadata:
      labels:
        app: {name}
        tier: {tier}
      annotations:
        prometheus.io/scrape: "true"
        prometheus.io/port: "{port}"
        prometheus.io/path: "/metrics"
    spec:
      containers:
      - name: {name}
        image: nginx:alpine
        ports:
        - containerPort: 80
        resources:
          requests: {{cpu: "25m", memory: "32Mi"}}
          limits:   {{cpu: "100m", memory: "128Mi"}}
        livenessProbe:
          httpGet: {{path: "/", port: 80}}
          initialDelaySeconds: 5
          periodSeconds: 10
        readinessProbe:
          httpGet: {{path: "/", port: 80}}
          initialDelaySeconds: 3
          periodSeconds: 5
---
apiVersion: v1
kind: Service
metadata:
  name: {name}
  namespace: {NAMESPACE}
spec:
  selector:
    app: {name}
  ports:
  - port: {port}
    targetPort: 80
"""

manifest_all = "".join(build_manifest(*s) for s in SERVICES)
with open("/tmp/lab8-services.yaml", "w") as f:
    f.write(manifest_all)

print("Applying manifests...")
run("kubectl apply -f /tmp/lab8-services.yaml")
print()
print("⏳ Waiting 30 seconds for pods to initialise...")
time.sleep(30)
print("Done waiting.")

In [ ]:
# ── Verify deployment ──────────────────────────────────────────────────────────
# All pods should show STATUS = Running before you continue.

print("PODS in namespace aiops-lab8:")
print("-" * 70)
run(f"kubectl get pods -n {NAMESPACE} -o wide")

print("\nSERVICES in namespace aiops-lab8:")
print("-" * 70)
run(f"kubectl get services -n {NAMESPACE}")

print()
print("💡 If any pods show 'Pending' or 'ContainerCreating', wait 30 more")
print("   seconds and re-run this cell. Do NOT continue until all are Running.")

---
## Phase 2 — Simulate an Alert Storm 🌩️

### 📖 Concept: Alert Storms & Alert Fatigue

> An **alert storm** occurs when a single failure event (e.g., a database going down) triggers a *cascade* of alerts from every service that depends on it — directly or transitively. The operations team receives hundreds of notifications within minutes, all seeming equally urgent. This is **alert fatigue**.

**Real-world scale:** Netflix reported ~14,000 alerts from one CDN failure. AWS postmortems consistently show that operators spend the first 20–30 minutes *filtering alerts* rather than fixing the problem.

### What we simulate:

- **T = -10min → T=0** : Normal baseline noise (random low-severity alerts)
- **T = 0** : `inventory-service` database connection pool exhausted → CRITICAL
- **T = 0 → T+5min** : Cascade propagates — every dependent service starts firing
- **Throughout** : Each alert fires repeatedly every 30s (Prometheus default scrape interval) = **duplicates**

In [ ]:
# ── Alert data model & constants ───────────────────────────────────────────────

SEVERITIES = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "INFO"]
SEV_RANK   = {s: i for i, s in enumerate(SEVERITIES)}   # lower = worse

ALERT_CATALOG = {
    "availability":  ["ServiceDown",             "PodCrashLooping",     "HealthCheckFailing"],
    "error_rate":    ["HTTP5xxSpike",             "ErrorBudgetBurn",     "DependencyErrors"],
    "latency":       ["HighP99Latency",           "SlowResponseTime",    "TimeoutRateHigh"],
    "saturation":    ["ConnectionPoolExhausted",  "CPUThrottling",       "MemoryPressure"],
    "resource":      ["DiskSpaceLow",             "NetworkSaturation",   "IOWaitHigh"],
}

INCIDENT_START = datetime(2025, 6, 15, 14, 0, 0)   # T=0

def make_alert(service, category, severity, ts, root_cause=None):
    name = random.choice(ALERT_CATALOG[category])
    fp_raw = f"{service}|{name}|{severity}"
    return {
        "alert_id":   "ALT-" + hashlib.md5(f"{fp_raw}{ts}".encode()).hexdigest()[:8].upper(),
        "timestamp":  ts,
        "service":    service,
        "alert_name": name,
        "category":   category,
        "severity":   severity,
        "root_cause": root_cause,
        "message":    f"{name} on {service} — threshold breached",
        "fingerprint": hashlib.md5(fp_raw.encode()).hexdigest()[:10],
    }

print("✅ Alert model ready")
print(f"   Catalog: {sum(len(v) for v in ALERT_CATALOG.values())} unique alert types across {len(ALERT_CATALOG)} categories")

In [ ]:
# ── Generate the storm ─────────────────────────────────────────────────────────
all_alerts = []

# ── Segment A: Baseline noise (T = -10min → T=0) ─────────────────────────────
print("Generating baseline noise (pre-incident)...")
for _ in range(45):
    svc = random.choice(list(DEPENDENCY_GRAPH.keys()))
    cat = random.choice(["latency", "resource", "saturation"])
    sev = random.choice(["LOW", "INFO", "MEDIUM"])
    ts  = INCIDENT_START - timedelta(seconds=random.randint(30, 600))
    all_alerts.append(make_alert(svc, cat, sev, ts))

# ── Segment B: Root cause fires at T=0 (inventory-service DB failure) ─────────
print("Triggering root cause: inventory-service DB failure...")
for burst in range(10):
    ts = INCIDENT_START + timedelta(seconds=burst * 3)
    all_alerts.append(make_alert("inventory-service", "saturation",  "CRITICAL", ts, root_cause="inventory-service"))
    all_alerts.append(make_alert("inventory-service", "availability","CRITICAL", ts, root_cause="inventory-service"))
    all_alerts.append(make_alert("inventory-service", "error_rate",  "HIGH",     ts, root_cause="inventory-service"))

# ── Segment C: Cascade — downstream services start failing (T+20s → T+5min) ───
print("Propagating cascade through dependent services...")

cascade = {
    #  service                  start_sec  end_sec  categories               sevs
    "order-service":        (20,  280, ["error_rate","latency","availability"], ["CRITICAL","HIGH","HIGH"]),
    "payment-service":      (25,  280, ["error_rate","latency"],               ["HIGH","HIGH","MEDIUM"]),
    "fraud-service":        (30,  260, ["latency","availability"],             ["HIGH","MEDIUM"]),
    "api-gateway":          (50,  300, ["error_rate","latency","saturation"],  ["CRITICAL","HIGH","HIGH"]),
    "notification-service": (55,  240, ["error_rate","availability"],         ["MEDIUM","LOW"]),
    "frontend":             (75,  300, ["latency","error_rate"],               ["HIGH","MEDIUM"]),
}

for svc, (t_start, t_end, cats, sevs) in cascade.items():
    t = t_start
    while t < t_end:
        cat = random.choice(cats)
        sev = random.choice(sevs)
        ts  = INCIDENT_START + timedelta(seconds=t)
        all_alerts.append(make_alert(svc, cat, sev, ts, root_cause="inventory-service"))
        # Simulate Prometheus re-firing the same alert every 30s
        for dup_offset in [30, 60, 90]:
            if t + dup_offset < t_end:
                ts_dup = INCIDENT_START + timedelta(seconds=t + dup_offset)
                all_alerts.append(make_alert(svc, cat, sev, ts_dup, root_cause="inventory-service"))
        t += random.randint(8, 20)

# ── Segment D: Unrelated coincidental alerts ───────────────────────────────────
print("Adding unrelated coincidental alerts (noise)...")
for _ in range(25):
    svc = random.choice(["frontend", "api-gateway"])
    ts  = INCIDENT_START + timedelta(seconds=random.randint(0, 300))
    all_alerts.append(make_alert(svc, "resource", random.choice(["MEDIUM","LOW"]), ts))

# Sort by time
all_alerts.sort(key=lambda x: x["timestamp"])
df_raw = pd.DataFrame(all_alerts)

print()
print("=" * 55)
print(f"  Total raw alerts generated  : {len(df_raw):>5}")
print(f"  Storm alerts (cascaded)     : {df_raw['root_cause'].notna().sum():>5}")
print(f"  Baseline / coincidental     : {df_raw['root_cause'].isna().sum():>5}")
print(f"  Unique fingerprints         : {df_raw['fingerprint'].nunique():>5}")
print("=" * 55)

In [ ]:
# ── Visualise the Alert Storm ──────────────────────────────────────────────────
# This chart shows WHY alert storms cause fatigue:
# - Volume overwhelms operators within the first 60 seconds
# - Every service looks equally broken
# - The real root cause is buried under the noise

df_raw["ts_sec"] = (df_raw["timestamp"] - INCIDENT_START).dt.total_seconds()
df_raw["bin30"]  = (df_raw["ts_sec"] // 30).astype(int)   # 30-second bins

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle(
    "Phase 2 — The Alert Storm: inventory-service DB Failure Cascading Through 7 Services",
    fontsize=13, fontweight="bold"
)

SEV_COLORS = {"CRITICAL": "#c0392b", "HIGH": "#e67e22",
              "MEDIUM":   "#f1c40f", "LOW":  "#27ae60", "INFO": "#3498db"}

# ── Chart 1: Alert volume over time (30s bins) ────────────────────────────────
ax = axes[0, 0]
timeline = df_raw.groupby("bin30").size().reset_index(name="count")
ax.bar(timeline["bin30"] * 0.5, timeline["count"],
       width=0.45, color="#c0392b", alpha=0.85)
ax.axvline(0, color="black", linestyle="--", linewidth=2, label="T=0: DB failure")
ax.set_xlabel("Minutes from incident start")
ax.set_ylabel("Alerts per 30 seconds")
ax.set_title("🌩️ Alert Volume — Operator Cannot Process This Rate")
ax.legend()

# ── Chart 2: Stacked severity per service ─────────────────────────────────────
ax2 = axes[0, 1]
svc_sev = (df_raw.groupby(["service", "severity"])
               .size().unstack(fill_value=0)
               .reindex(columns=SEVERITIES, fill_value=0))
colors2 = [SEV_COLORS[s] for s in svc_sev.columns]
svc_sev.plot(kind="bar", stacked=True, ax=ax2,
             color=colors2, width=0.7, edgecolor="white", linewidth=0.5)
ax2.set_title("⚡ Alerts per Service — All Look Equally 'On Fire'")
ax2.set_xlabel("Service")
ax2.set_ylabel("Alert Count")
ax2.legend(title="Severity", fontsize=8, bbox_to_anchor=(1.01,1), loc="upper left")
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=30, ha="right", fontsize=8)

# ── Chart 3: Category distribution ───────────────────────────────────────────
ax3 = axes[1, 0]
cat_counts = df_raw["category"].value_counts()
ax3.pie(cat_counts.values, labels=cat_counts.index, autopct="%1.0f%%",
        startangle=140,
        colors=["#e74c3c","#e67e22","#3498db","#2ecc71","#9b59b6"])
ax3.set_title("Alert Categories — Spread Across All Types")

# ── Chart 4: Cumulative alert count ───────────────────────────────────────────
ax4 = axes[1, 1]
times_sorted = sorted(df_raw["ts_sec"].values)
ax4.plot([t/60 for t in times_sorted], range(1, len(times_sorted)+1),
         color="#c0392b", linewidth=2)
ax4.axvline(0, color="black", linestyle="--", linewidth=2, label="Incident start")
ax4.fill_between([t/60 for t in times_sorted],
                 range(1, len(times_sorted)+1),
                 alpha=0.15, color="#c0392b")
ax4.set_xlabel("Minutes from incident start")
ax4.set_ylabel("Cumulative alerts")
ax4.set_title(f"📈 Cumulative Growth — {len(df_raw)} Alerts in ~6 Minutes")
ax4.legend()

plt.tight_layout()
plt.savefig("/tmp/phase2_alert_storm.png", dpi=120, bbox_inches="tight")
plt.show()

print()
print("💡 KEY INSIGHT:")
print(f"   An on-call engineer receives {len(df_raw)} alerts in ~6 minutes.")
print("   They cannot determine the root cause from raw alerts alone.")
print("   This is the problem AIOps solves — starting in Phase 3.")

---
## Phase 3 — Alert Deduplication & Noise Reduction 🔇

### 📖 Concept: Why Deduplication is the First Step

Monitoring systems like Prometheus fire the same alert **every scrape interval** (typically 30s) until the condition clears. This means:
- 1 real problem → potentially **hundreds of identical alert events**
- Signal-to-noise ratio collapses
- On-call queues fill with duplicates

### Techniques we apply:

| Technique | Logic |
|-----------|-------|
| **Fingerprinting** | Hash `service + alert_name + severity` → unique identity per alert type |
| **Time-window dedup** | Suppress re-fires of the same fingerprint within a 5-minute window |
| **Flap detection** | Flag alerts firing ≥ 3 times within 10 minutes as *flapping* (intermittent) |
| **Severity normalisation** | Downgrade repeated non-critical alerts to avoid over-paging |

In [ ]:
# ── Step 3a: Inspect fingerprints ─────────────────────────────────────────────
# Each fingerprint represents a UNIQUE ALERT TYPE (service + name + severity).
# High fire_count = that alert is spamming the queue.

fp_summary = (
    df_raw.groupby(["fingerprint", "service", "alert_name", "severity"])
    .size()
    .reset_index(name="fire_count")
    .sort_values("fire_count", ascending=False)
)

print(f"Unique fingerprints (distinct alert types): {len(fp_summary)}")
print(f"Total raw alert events                    : {len(df_raw)}")
print(f"Average duplicates per alert type         : {len(df_raw)/len(fp_summary):.1f}x")
print()
print("Top 12 most spammy alert fingerprints:")
print(tabulate(
    fp_summary.head(12),
    headers=["Fingerprint", "Service", "Alert Name", "Severity", "Times Fired"],
    tablefmt="rounded_outline",
    showindex=False
))

In [ ]:
# ── Step 3b: Time-window deduplication ────────────────────────────────────────
# Rule: For each fingerprint, keep the FIRST occurrence within each 5-min window.
# Subsequent firings of the same fingerprint within the window → suppressed.

DEDUP_WINDOW_SEC = 300   # 5 minutes

def deduplicate(df, window_seconds):
    df = df.sort_values("timestamp").copy()
    last_kept  = {}   # fingerprint → timestamp of last kept event
    keep       = []

    for _, row in df.iterrows():
        fp = row["fingerprint"]
        ts = row["timestamp"]
        if fp not in last_kept or (ts - last_kept[fp]).total_seconds() >= window_seconds:
            keep.append(True)
            last_kept[fp] = ts
        else:
            keep.append(False)

    df["kept"] = keep
    return df[df["kept"]].drop(columns=["kept"]).copy()

df_dedup = deduplicate(df_raw, DEDUP_WINDOW_SEC)

n_raw   = len(df_raw)
n_dedup = len(df_dedup)
pct_removed = (1 - n_dedup / n_raw) * 100

print(f"Deduplication window : {DEDUP_WINDOW_SEC}s ({DEDUP_WINDOW_SEC//60} minutes)")
print()
print(f"  Raw alerts         : {n_raw:>5}")
print(f"  After dedup        : {n_dedup:>5}")
print(f"  Suppressed         : {n_raw - n_dedup:>5}  ({pct_removed:.1f}% noise removed)")
print()
print("✅ Over half the alert volume removed with one simple rule.")

In [ ]:
# ── Step 3c: Flap detection ────────────────────────────────────────────────────
# A 'flapping' alert fires, clears, fires, clears repeatedly.
# It indicates an intermittent condition — important to detect so operators
# don't keep responding to it as if it's a new event each time.

FLAP_WINDOW_SEC = 600   # 10 minutes
FLAP_THRESHOLD  = 3     # fires at least 3 times in the window

def detect_flapping(df, window_sec, threshold):
    df = df.copy()
    df["flapping"] = False
    window = timedelta(seconds=window_sec)

    for fp in df["fingerprint"].unique():
        rows  = df[df["fingerprint"] == fp].sort_values("timestamp")
        times = rows["timestamp"].tolist()
        for t in times:
            count_in_window = sum(
                1 for tt in times
                if timedelta(0) <= (tt - t) <= window
            )
            if count_in_window >= threshold:
                df.loc[(df["fingerprint"] == fp) & (df["timestamp"] == t), "flapping"] = True
    return df

df_dedup = detect_flapping(df_dedup, FLAP_WINDOW_SEC, FLAP_THRESHOLD)
n_flapping = df_dedup["flapping"].sum()

print(f"Flapping alerts detected: {n_flapping} ({n_flapping/len(df_dedup)*100:.1f}% of deduped alerts)")
print()
if n_flapping > 0:
    flap_table = (
        df_dedup[df_dedup["flapping"]]
        [["service","alert_name","severity","category"]]
        .drop_duplicates()
        .head(10)
    )
    print("Flapping alert types (intermittent — lower priority):")
    print(tabulate(flap_table, headers="keys", tablefmt="rounded_outline", showindex=False))
    print()
    print("💡 These will be flagged in the incident register but NOT page on-call.")

In [ ]:
# ── Visualise: Before vs After Noise Reduction ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Phase 3 — Alert Noise Reduction Results",
             fontsize=13, fontweight="bold")

# Panel 1: Funnel
ax = axes[0]
stages = ["Raw Alerts", "After Dedup", "Flapping"]
values = [n_raw, n_dedup, n_flapping]
bar_colors = ["#c0392b", "#e67e22", "#f1c40f"]
bars = ax.bar(stages, values, color=bar_colors, edgecolor="white", linewidth=1.5, width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha="center", fontweight="bold", fontsize=12)
ax.set_title("Alert Volume Reduction")
ax.set_ylabel("Count")

# Panel 2: Raw severity distribution
ax2 = axes[1]
raw_sev = df_raw["severity"].value_counts().reindex(SEVERITIES, fill_value=0)
ax2.bar(raw_sev.index, raw_sev.values,
        color=[SEV_COLORS[s] for s in raw_sev.index],
        edgecolor="white")
ax2.set_title("Severity: RAW")
ax2.set_ylabel("Count")

# Panel 3: Deduped severity distribution
ax3 = axes[2]
ded_sev = df_dedup["severity"].value_counts().reindex(SEVERITIES, fill_value=0)
ax3.bar(ded_sev.index, ded_sev.values,
        color=[SEV_COLORS[s] for s in ded_sev.index],
        edgecolor="white")
ax3.set_title("Severity: AFTER DEDUP")
ax3.set_ylabel("Count")

plt.tight_layout()
plt.savefig("/tmp/phase3_noise_reduction.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\n📊 Summary: {n_raw} raw alerts → {n_dedup} actionable alerts ({pct_removed:.0f}% noise removed)")

---
## Phase 4 — Event Correlation Techniques 🔗

### 📖 Concept: Event Correlation

After deduplication, we still have many alerts from many services. **Event correlation** answers the question:

> *Are these alerts caused by the same underlying event?*

We implement **three complementary techniques** — each catches what the others miss:

| Technique | Strength | Weakness |
|-----------|----------|----------|
| **Rule-Based (Topology)** | Precise, explainable | Requires up-to-date dependency map |
| **Temporal (DBSCAN)** | Catches unknown relations | Can over-cluster unrelated events |
| **Causal (Time-lag)** | Finds cause→effect sequences | Needs sufficient data |

> In production systems (PagerDuty, Moogsoft, IBM AIOps) all three are used in combination.

In [ ]:
# ── Technique 4a: Rule-Based Topology Correlation ─────────────────────────────
# Rule: If service X has a CRITICAL/HIGH alert AND service Y depends on X
#       AND Y fires within CORR_WINDOW seconds → mark Y's alert as correlated
#       to X's alert.
# This directly exploits the known service dependency graph.

CORR_WINDOW_SEC = 150   # 2.5-minute correlation window

def get_transitive_dependents(service, graph):
    """Who calls 'service'? (direct and transitive callers)"""
    dependents = set()
    for svc, deps in graph.items():
        if service in deps:
            dependents.add(svc)
            dependents.update(get_transitive_dependents(svc, graph))
    return dependents

df_corr = df_dedup.copy()
df_corr["corr_group"]    = None
df_corr["corr_cause"]    = None
df_corr["corr_method"]   = "uncorrelated"
df_corr["ts_sec"] = (df_corr["timestamp"] - INCIDENT_START).dt.total_seconds()

group_counter = 1

# Process CRITICAL alerts as potential root causes, in chronological order
critical_alerts = df_corr[df_corr["severity"] == "CRITICAL"].sort_values("timestamp")

for _, cause in critical_alerts.iterrows():
    cause_svc = cause["service"]
    cause_ts  = cause["timestamp"]
    grp_id    = f"GRP-{group_counter:03d}"

    # Find services that would be affected if cause_svc fails
    affected_svcs = get_transitive_dependents(cause_svc, DEPENDENCY_GRAPH)
    if not affected_svcs:
        continue

    # Tag downstream alerts within the correlation window
    mask = (
        df_corr["service"].isin(affected_svcs)
        & (df_corr["timestamp"] > cause_ts)
        & (df_corr["timestamp"] <= cause_ts + timedelta(seconds=CORR_WINDOW_SEC))
        & (df_corr["corr_group"].isna())
    )

    if mask.sum() > 0:
        df_corr.loc[mask, "corr_group"]  = grp_id
        df_corr.loc[mask, "corr_cause"]  = cause_svc
        df_corr.loc[mask, "corr_method"] = "rule-topology"
        # Tag the cause alert itself
        df_corr.loc[df_corr["alert_id"] == cause["alert_id"], "corr_group"]  = grp_id
        df_corr.loc[df_corr["alert_id"] == cause["alert_id"], "corr_cause"]  = cause_svc
        df_corr.loc[df_corr["alert_id"] == cause["alert_id"], "corr_method"] = "rule-topology"
        group_counter += 1

corr_count = df_corr["corr_group"].notna().sum()
print(f"Rule-based correlation:")
print(f"  Alerts correlated : {corr_count} / {len(df_corr)}")
print(f"  Groups created    : {group_counter - 1}")
print(f"  Coverage          : {corr_count/len(df_corr)*100:.0f}%")

In [ ]:
# ── Technique 4b: Temporal Clustering with DBSCAN ─────────────────────────────
# DBSCAN (Density-Based Spatial Clustering of Applications with Noise) groups
# points that are densely packed together. Here we cluster in TIME — alerts
# that occur close together in time are likely related.
#
# Applied to: alerts still UNCORRELATED after rule-based pass.

uncorrelated = df_corr[df_corr["corr_group"].isna()].copy()
print(f"Uncorrelated alerts remaining: {len(uncorrelated)}")

if len(uncorrelated) >= 4:
    svc_idx = {s: i for i, s in enumerate(sorted(df_corr["service"].unique()))}
    uncorrelated["svc_idx"] = uncorrelated["service"].map(svc_idx)

    # Features: normalised (time, service_index)
    X = StandardScaler().fit_transform(
        uncorrelated[["ts_sec", "svc_idx"]].fillna(0)
    )

    db = DBSCAN(eps=0.5, min_samples=2).fit(X)
    uncorrelated = uncorrelated.copy()
    uncorrelated["dbscan_label"] = db.labels_

    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise    = (db.labels_ == -1).sum()
    print(f"DBSCAN result: {n_clusters} clusters, {n_noise} noise points")

    for clabel in sorted(set(db.labels_)):
        if clabel == -1:
            continue
        grp_id = f"GRP-DBSCAN-{clabel:03d}"
        idxs   = uncorrelated[uncorrelated["dbscan_label"] == clabel].index
        df_corr.loc[idxs, "corr_group"]  = grp_id
        df_corr.loc[idxs, "corr_method"] = "dbscan-temporal"

    print(f"DBSCAN correlated {(df_corr['corr_method']=='dbscan-temporal').sum()} additional alerts")
else:
    print("Not enough uncorrelated alerts for DBSCAN — rule-based covered everything.")

total_corr = df_corr["corr_group"].notna().sum()
print()
print(f"Total correlated (all methods): {total_corr} / {len(df_corr)} ({total_corr/len(df_corr)*100:.0f}%)")  

In [ ]:
# ── Technique 4c: Causal Time-Lag Analysis ────────────────────────────────────
# Measure HOW LONG after the root-cause alert each downstream service first fires.
# This builds a 'propagation timeline' — useful for SLA analysis and capacity planning.

root_cause_svc = "inventory-service"
root_first_ts  = (
    df_corr[df_corr["service"] == root_cause_svc]["timestamp"].min()
)

propagation = []
for svc in [s[0] for s in SERVICES]:
    svc_alerts = df_corr[df_corr["service"] == svc]
    if len(svc_alerts) == 0:
        continue
    first_alert_ts = svc_alerts["timestamp"].min()
    lag_sec        = (first_alert_ts - root_first_ts).total_seconds()
    propagation.append({
        "service":      svc,
        "first_alert":  first_alert_ts.strftime("%H:%M:%S"),
        "lag_seconds":  round(lag_sec, 1),
        "alert_count":  len(svc_alerts),
        "worst_sev":    svc_alerts["severity"].map(SEV_RANK).idxmin(),
    })

# Fix worst_sev to show the severity label
for p in propagation:
    idx = p["worst_sev"]
    p["worst_sev"] = df_corr.loc[idx, "severity"] if idx in df_corr.index else "?"

df_prop = pd.DataFrame(propagation).sort_values("lag_seconds")

print("Failure Propagation Timeline:")
print(tabulate(
    df_prop,
    headers=["Service", "First Alert", "Lag (sec)", "Alert Count", "Worst Severity"],
    tablefmt="rounded_outline",
    showindex=False
))

# Visualise propagation
fig, ax = plt.subplots(figsize=(11, 4))
df_prop_sorted = df_prop.sort_values("lag_seconds")
colors_p = [SEV_COLORS.get(s, "gray") for s in df_prop_sorted["worst_sev"]]
bars = ax.barh(df_prop_sorted["service"], df_prop_sorted["lag_seconds"],
               color=colors_p, edgecolor="white", height=0.6)
ax.axvline(0, color="black", linestyle="--", linewidth=1.5)
for bar, val in zip(bars, df_prop_sorted["lag_seconds"]):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f"+{val:.0f}s", va="center", fontsize=9)
ax.set_xlabel("Seconds after root cause fired")
ax.set_title("Failure Propagation — How Long Until Each Service Was Impacted")
handles = [mpatches.Patch(color=SEV_COLORS[s], label=s) for s in SEVERITIES if s in SEV_COLORS]
ax.legend(handles=handles, title="Worst Severity", loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/phase4_propagation.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── Correlation methods summary ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Phase 4 — Event Correlation Results", fontsize=13, fontweight="bold")

# Panel 1: Methods breakdown
ax = axes[0]
meth_counts = df_corr["corr_method"].value_counts()
ax.pie(meth_counts.values, labels=meth_counts.index,
       autopct="%1.0f%%",
       colors=["#3498db","#e67e22","#2ecc71","#95a5a6"][:len(meth_counts)],
       startangle=130)
ax.set_title("How Alerts Were Correlated")

# Panel 2: Group sizes (top 10 groups)
ax2 = axes[1]
grp_sizes = (
    df_corr[df_corr["corr_group"].notna()]
    .groupby("corr_group").size()
    .sort_values(ascending=False)
    .head(10)
)
ax2.bar(range(len(grp_sizes)), grp_sizes.values,
        color="#3498db", edgecolor="white")
ax2.set_xticks(range(len(grp_sizes)))
ax2.set_xticklabels(grp_sizes.index, rotation=30, ha="right", fontsize=8)
ax2.set_title("Top 10 Correlation Groups by Size")
ax2.set_ylabel("Alerts in group")

plt.tight_layout()
plt.savefig("/tmp/phase4_correlation.png", dpi=120, bbox_inches="tight")
plt.show()

print()
method_breakdown = df_corr["corr_method"].value_counts().to_dict()
for method, count in method_breakdown.items():
    print(f"  {method:25s}: {count:>4} alerts")

---
## Phase 5 — Service Dependency Mapping & Root Cause Analysis 🗺️

### 📖 Concept: Why Dependency Maps are Essential

> Without a dependency map, **all alerts look equally important**. With a dependency map, we can automatically determine:
> - Which alerts are **symptoms** (downstream effects)
> - Which alert is the **cause** (the root of the failure tree)

### Root Cause Analysis Algorithm

We use **graph traversal** on the service dependency graph:

1. Build a directed graph `A → B` meaning "A calls B" (A depends on B)
2. Annotate each node with its current worst alert severity
3. Walk the graph: a node is a **root cause candidate** if:
   - It has HIGH or CRITICAL alerts
   - None of its *dependencies* (successors) are in a worse state
   - It is NOT itself failing because of something it calls

In [ ]:
# ── Build the annotated dependency graph ──────────────────────────────────────
G = nx.DiGraph()

for svc, data in [(s[0], {"tier": s[3]}) for s in SERVICES]:
    svc_alerts = df_corr[df_corr["service"] == svc]
    if len(svc_alerts) == 0:
        worst_sev = "OK"
        worst_rank = 99
    else:
        ranked = svc_alerts["severity"].map(SEV_RANK)
        worst_sev = svc_alerts.loc[ranked.idxmin(), "severity"]
        worst_rank = ranked.min()

    G.add_node(svc,
               tier=data["tier"],
               severity=worst_sev,
               sev_rank=worst_rank,
               alert_count=len(svc_alerts))

# Edges: caller → callee
for svc, deps in DEPENDENCY_GRAPH.items():
    for dep in deps:
        G.add_edge(svc, dep)

print(f"Dependency graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print()
print("Node state after Phase 3+4 processing:")
print(tabulate(
    [(n, G.nodes[n]["severity"], G.nodes[n]["alert_count"],
      G.nodes[n]["tier"], ", ".join(list(G.predecessors(n))) or "(none)")
     for n in G.nodes()],
    headers=["Service", "Worst Severity", "Alerts", "Tier", "Called By"],
    tablefmt="rounded_outline"
))

In [ ]:
# ── Root Cause Analysis via Graph Traversal ───────────────────────────────────

def root_cause_analysis(G):
    candidates = []
    for node in G.nodes():
        node_rank = G.nodes[node]["sev_rank"]
        node_sev  = G.nodes[node]["severity"]

        # Only consider nodes with actual problems
        if node_rank >= SEV_RANK["MEDIUM"]:   # MEDIUM, LOW, INFO, OK → skip
            continue

        # Look at what this node CALLS (its dependencies)
        deps = list(G.successors(node))
        dep_ranks = [G.nodes[d]["sev_rank"] for d in deps]

        # RCA rule: I am a root cause if none of my dependencies are in WORSE shape
        # (i.e., I am not just a victim of a downstream failure)
        if not dep_ranks or min(dep_ranks) > node_rank:
            # Double-check: are any dependents (services that call ME) failing?
            callers = list(G.predecessors(node))
            impacted = [c for c in callers if G.nodes[c]["sev_rank"] < SEV_RANK["MEDIUM"]]

            candidates.append({
                "service":    node,
                "severity":   node_sev,
                "alert_count": G.nodes[node]["alert_count"],
                "impacted":   impacted,
                "confidence": "HIGH" if not dep_ranks else "MEDIUM",
            })

    return sorted(candidates, key=lambda x: SEV_RANK.get(x["severity"], 99))

rca_results = root_cause_analysis(G)

print("═" * 60)
print("  🎯  ROOT CAUSE ANALYSIS — RESULTS")
print("═" * 60)
for i, rc in enumerate(rca_results, 1):
    print(f"\n  #{i}  Root Cause Candidate: {rc['service'].upper()}")
    print(f"       Severity   : {rc['severity']}")
    print(f"       Alerts     : {rc['alert_count']}")
    print(f"       Confidence : {rc['confidence']}")
    if rc['impacted']:
        print(f"       Impacted   : {', '.join(rc['impacted'])}")
    else:
        print(f"       Impacted   : (leaf node — no callers affected or callers ok)")
print()
print("═" * 60)
if rca_results:
    top = rca_results[0]
    print(f"  ✅  TOP CANDIDATE: {top['service']} (Confidence: {top['confidence']})")
    print(f"  ✅  This matches our simulated failure: inventory-service DB exhaustion")
print("═" * 60)

In [ ]:
# ── Draw the Dependency Map with Live Alert Overlay ───────────────────────────

fig, ax = plt.subplots(figsize=(14, 8))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

# Fixed layout matching the architecture diagram
pos = {
    "frontend":             (0.0, 2.5),
    "api-gateway":          (2.0, 2.5),
    "order-service":        (4.0, 3.8),
    "payment-service":      (4.0, 2.5),
    "notification-service": (4.0, 1.2),
    "inventory-service":    (6.5, 3.8),
    "fraud-service":        (6.5, 2.5),
}

NODE_COLORS_DARK = {
    "CRITICAL": "#c0392b",
    "HIGH":     "#d35400",
    "MEDIUM":   "#d4ac0d",
    "LOW":      "#1e8449",
    "INFO":     "#1a5276",
    "OK":       "#1a5276",
}

node_colors = [NODE_COLORS_DARK.get(G.nodes[n]["severity"], "#555") for n in G.nodes()]
node_sizes  = [1200 + G.nodes[n]["alert_count"] * 50 for n in G.nodes()]

nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color="#4a5568", arrows=True, arrowsize=25,
    width=2.5, connectionstyle="arc3,rad=0.08",
    arrowstyle="-|>"
)
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=node_colors, node_size=node_sizes, alpha=0.93
)

# Labels inside nodes
node_labels = {
    n: f"{n}\n[{G.nodes[n]['severity']}]\n{G.nodes[n]['alert_count']} alerts"
    for n in G.nodes()
}
nx.draw_networkx_labels(
    G, pos, labels=node_labels, ax=ax,
    font_color="white", font_size=7, font_weight="bold"
)

# Root cause annotation
if rca_results:
    rc_svc = rca_results[0]["service"]
    x, y   = pos[rc_svc]
    ax.annotate(
        "⚡ ROOT CAUSE\n(RCA identified)",
        xy=(x, y), xytext=(x + 0.4, y + 0.9),
        fontsize=9, color="#ffd700", fontweight="bold",
        arrowprops=dict(arrowstyle="->", color="#ffd700", lw=2)
    )

# Legend
legend_patches = [
    mpatches.Patch(color=NODE_COLORS_DARK[s], label=s)
    for s in ["CRITICAL", "HIGH", "MEDIUM", "OK"]
]
ax.legend(
    handles=legend_patches, title="Alert Status",
    loc="lower left", facecolor="#1a202c",
    edgecolor="#4a5568", labelcolor="white",
    title_fontsize=9, fontsize=8
)

ax.set_title(
    "Service Dependency Map — Live Alert State Overlay\n"
    "Node size ∝ alert count · Colour = worst severity · Arrow = dependency direction",
    color="white", fontsize=11, fontweight="bold", pad=12
)
ax.axis("off")
plt.tight_layout()
plt.savefig("/tmp/phase5_dependency_map.png", dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

print("\n💡 READING THE MAP:")
print("   - Arrows point FROM caller TO callee (dependency direction)")
print("   - Red nodes = CRITICAL alerts · Orange = HIGH")
print("   - Larger nodes = more alerts (but this doesn't mean more important!)")
print("   - The ROOT CAUSE is the red node with no red dependencies beneath it")

---
## Phase 6 — Incident Register: The Final AIOps Output 📋

### 📖 Concept: Alert → Incident Compression

The **final goal** of the entire pipeline is to transform an unmanageable alert flood into a concise, prioritised list of **incidents** that an operator can take action on.

Each incident contains:
- **What broke** (root cause service)
- **What's affected** (blast radius)
- **How urgent** (SLA tier)
- **What to do** (recommended remediation)
- **Evidence** (correlated alerts that justify the incident)

In [ ]:
# ── Build Incident Register ────────────────────────────────────────────────────

REMEDIATION_KB = {
    "inventory-service":
        "Check DB connection pool. Run: kubectl logs -n aiops-lab8 deploy/inventory-service\n"
        "     Verify: kubectl describe pod -n aiops-lab8 -l app=inventory-service",
    "order-service":
        "Verify inventory-service is healthy first. Check circuit-breaker state.\n"
        "     Run: kubectl exec -n aiops-lab8 deploy/order-service -- curl inventory-service:8083/health",
    "payment-service":
        "Check fraud-service and payment gateway API. Verify TLS certificates.",
    "api-gateway":
        "Check upstream service health. Validate rate limit config.\n"
        "     Run: kubectl top pods -n aiops-lab8",
    "frontend":
        "Check api-gateway health. Review CDN/ingress configuration.",
}

SLA_TIERS = {"CRITICAL": "P1 — 15 min", "HIGH": "P2 — 1 hr",
             "MEDIUM":   "P3 — 4 hr",   "LOW":  "P4 — 24 hr"}

incidents = []
grouped = df_corr[df_corr["corr_group"].notna()].groupby("corr_group")

for grp_id, grp in grouped:
    # Worst severity in this group
    worst_rank = grp["severity"].map(SEV_RANK).min()
    worst_sev  = [s for s, r in SEV_RANK.items() if r == worst_rank][0]

    # Root cause
    cause_col = grp["corr_cause"].dropna()
    root_cause = cause_col.mode().iloc[0] if len(cause_col) > 0 else "unknown"

    affected_svcs = sorted(grp["service"].unique().tolist())
    started_at    = grp["timestamp"].min()
    duration_min  = round((grp["timestamp"].max() - started_at).total_seconds() / 60, 1)

    incidents.append({
        "Incident ID":   grp_id,
        "Severity":      worst_sev,
        "Root Cause":    root_cause,
        "Blast Radius":  len(affected_svcs),
        "Raw Alerts":    len(grp),
        "Duration (min)": duration_min,
        "SLA":           SLA_TIERS.get(worst_sev, "P4"),
        "Started":       started_at.strftime("%H:%M:%S"),
        "Action":        REMEDIATION_KB.get(root_cause, "Check service logs and upstream dependencies."),
    })

# Sort: CRITICAL first, then by alert count
df_inc = pd.DataFrame(incidents)
df_inc = df_inc.sort_values(
    ["Severity", "Raw Alerts"],
    key=lambda col: col.map(SEV_RANK) if col.name == "Severity" else col,
    ascending=[True, False]
).reset_index(drop=True)

print("=" * 70)
print(f"  INCIDENT REGISTER  —  {len(df_inc)} incidents from {len(df_raw)} raw alerts")
print(f"  Compression ratio: {len(df_raw) // max(len(df_inc), 1)}x")
print("=" * 70)
print()
print(tabulate(
    df_inc[["Incident ID", "Severity", "Root Cause", "Blast Radius", "Raw Alerts", "SLA", "Started"]],
    headers="keys",
    tablefmt="rounded_outline",
    showindex=False
))

In [ ]:
# ── Show the top incident with full remediation detail ────────────────────────
top = df_inc.iloc[0]

border = "═" * 65
print(border)
print(f"  🚨  INCIDENT: {top['Incident ID']}")
print(border)
print(f"  Severity        : {top['Severity']}")
print(f"  SLA             : {top['SLA']}")
print(f"  Root Cause      : {top['Root Cause']}")
print(f"  Blast Radius    : {top['Blast Radius']} service(s) impacted")
print(f"  Raw Alerts      : {top['Raw Alerts']} (compressed from storm)")
print(f"  Started At      : {top['Started']}")
print(f"  Duration        : {top['Duration (min)']} min")
print()
print("  📋  RECOMMENDED ACTION:")
for line in top['Action'].split('\n'):
    print(f"  {line}")
print(border)

In [ ]:
# ── Final Summary Dashboard ────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle("AIOps Lab 8 — Complete Pipeline Summary Dashboard",
             fontsize=15, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel 1: AIOps Reduction Funnel ───────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
stages = ["Raw\nAlerts", "After\nDedup", "Correlated", "Incidents"]
vals   = [len(df_raw), n_dedup,
          int(df_corr["corr_group"].notna().sum()), len(df_inc)]
cols   = ["#c0392b", "#e67e22", "#f1c40f", "#27ae60"]
bars   = ax1.bar(stages, vals, color=cols, edgecolor="white", linewidth=1.5, width=0.55)
for b, v in zip(bars, vals):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
             str(v), ha="center", fontweight="bold", fontsize=12)
ax1.set_title("AIOps Noise Reduction Funnel", fontweight="bold")
ax1.set_ylabel("Count")

# ── Panel 2: Incident severity pie ────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
inc_sev = df_inc["Severity"].value_counts()
ax2.pie(inc_sev.values, labels=inc_sev.index, autopct="%1.0f%%",
        colors=[SEV_COLORS.get(s, "gray") for s in inc_sev.index],
        startangle=90)
ax2.set_title("Incident Severity Distribution", fontweight="bold")

# ── Panel 3: Correlation methods used ─────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
meth = df_corr["corr_method"].value_counts()
ax3.barh(meth.index, meth.values,
         color=["#3498db","#9b59b6","#1abc9c","#95a5a6"][:len(meth)])
ax3.set_title("Correlation Methods Applied", fontweight="bold")
ax3.set_xlabel("Alerts")

# ── Panel 4: Raw vs Deduped per service ───────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
raw_ps  = df_raw.groupby("service").size().rename("raw")
ded_ps  = df_dedup.groupby("service").size().rename("deduped")
comp    = pd.concat([raw_ps, ded_ps], axis=1).fillna(0).astype(int)
x = np.arange(len(comp))
ax4.bar(x - 0.2, comp["raw"],    0.38, label="Raw",    color="#c0392b", alpha=0.85)
ax4.bar(x + 0.2, comp["deduped"],0.38, label="Deduped",color="#27ae60", alpha=0.85)
ax4.set_xticks(x)
ax4.set_xticklabels(comp.index, rotation=35, ha="right", fontsize=7)
ax4.set_title("Raw vs Deduped per Service", fontweight="bold")
ax4.legend(fontsize=8)

# ── Panel 5: Key Metrics Table ────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.axis("off")
metrics = [
    ["Raw alert events",        str(len(df_raw))],
    ["After deduplication",     str(n_dedup)],
    ["Noise removed",           f"{pct_removed:.0f}%"],
    ["Alerts correlated",       str(int(df_corr['corr_group'].notna().sum()))],
    ["Final incidents",         str(len(df_inc))],
    ["Compression ratio",       f"{len(df_raw)//max(len(df_inc),1)}x"],
    ["RCA: root cause found",   "✅ inventory-service"],
    ["RCA method",              "Topology graph traversal"],
]
tbl = ax5.table(cellText=metrics,
                colLabels=["Metric", "Value"],
                cellLoc="left", loc="center",
                colWidths=[0.58, 0.42])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.65)
ax5.set_title("Key AIOps Metrics", fontweight="bold", pad=12)

# ── Panel 6: Propagation timeline bar chart ───────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
df_prop_s = df_prop.sort_values("lag_seconds")
bars6 = ax6.barh(
    df_prop_s["service"], df_prop_s["lag_seconds"],
    color=[SEV_COLORS.get(s, "gray") for s in df_prop_s["worst_sev"]],
    edgecolor="white", height=0.6
)
ax6.set_xlabel("Seconds after root cause")
ax6.set_title("Failure Propagation Timeline", fontweight="bold")

plt.savefig("/tmp/phase6_final_dashboard.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n✅ Pipeline complete. See incident register above for remediation steps.")

---
## Phase 7 — Cleanup 🧹

Remove all Kubernetes resources created during this lab.

> ⚠️ This deletes the `aiops-lab8` namespace and everything inside it.

In [ ]:
# ── Delete the lab namespace ────────────────────────────────────────────────────
print("Deleting namespace aiops-lab8...")
out, rc = run("kubectl delete namespace aiops-lab8 --ignore-not-found=true")

if rc == 0:
    print("\n✅ Namespace deleted. All pods, services, and deployments removed.")
else:
    print("\n⚠️  Check manually: kubectl get all -n aiops-lab8")

---
## 📚 Lab Recap & Key Takeaways

### End-to-End Pipeline Recap

```
  ~500 Raw Alerts
       │
       ▼  Phase 3: Fingerprint + Dedup (5-min window)
  ~150 Unique Alerts  [~70% noise removed]
       │
       ▼  Phase 3: Flap Detection
  Flapping alerts annotated & deprioritised
       │
       ▼  Phase 4a: Rule-Based Topology Correlation
  Alerts grouped by service dependency graph
       │
       ▼  Phase 4b: DBSCAN Temporal Clustering
  Remaining alerts grouped by time proximity
       │
       ▼  Phase 5: Graph Traversal RCA
  Root cause = inventory-service (auto-identified)
       │
       ▼  Phase 6: Incident Compression
  ~10 Actionable Incidents  [~50x compression]
```

### 5 Core AIOps Principles You Practised

| # | Principle | Demonstrated By |
|---|-----------|----------------|
| 1 | **Alert ≠ Incident** | ~500 alerts → ~10 incidents |
| 2 | **Dedup first, correlate second** | 70% reduction before any ML |
| 3 | **Context beats volume** | Dependency map revealed root cause rules alone couldn't |
| 4 | **ML supplements rules** | DBSCAN caught alerts rule-topology missed |
| 5 | **Operator output = incident + action** | Every incident has an SLA and a kubectl command |

### Discussion Questions

1. What would happen if the dependency graph was **stale** or **wrong**? How do you keep it updated automatically?
2. How would you extend this pipeline to handle **multi-cluster** environments?
3. What additional **signals** (traces, logs, metrics) would improve correlation accuracy?
4. How do tools like **PagerDuty AIOps**, **Moogsoft**, and **IBM Watson AIOps** implement these techniques at scale?

### Further Reading

- [Google SRE Book — Alerting on SLOs](https://sre.google/workbook/alerting-on-slos/)
- [Prometheus Alerting Best Practices](https://prometheus.io/docs/practices/alerting/)
- [PagerDuty — Alert Noise Reduction](https://www.pagerduty.com/resources/learn/reduce-alert-noise/)
- [OpenTelemetry — Correlation Concepts](https://opentelemetry.io/docs/concepts/)

---

```
✅  Lab 8 Complete — AIOps Session 8: Noise Reduction & Alert Correlation
```